In [3]:
# Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Configurar la ruta a tu proyecto en RL
import sys
import os

# La ruta correcta según tu estructura de Drive
project_path = '/content/drive/MyDrive/MAESTRIA UROSARIO/Colab Notebooks/RL/Pacman_game'

# Agregar al path
sys.path.insert(0, project_path)
os.chdir(project_path)

print(f"Trabajando desde: {os.getcwd()}")
print(f"Archivos disponibles:\n{os.listdir()}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Trabajando desde: /content/drive/MyDrive/MAESTRIA UROSARIO/Colab Notebooks/RL/Pacman_game
Archivos disponibles:
['LICENSE', '.gitignore', 'test_cnn.py', 'dqn_cnn.py', 'README.md', 'test_pacman.py', 'Requirements.txt', 'notebook_entrenamiento.ipynb', '.github', '__pycache__', 'results', '.venv310', '.git', 'dqn_agent.py', 'train_dqn.py', 'wrappers.py']


In [4]:
import importlib
import wrappers
importlib.reload(wrappers)

<module 'wrappers' from '/content/drive/MyDrive/MAESTRIA UROSARIO/Colab Notebooks/RL/Pacman_game/wrappers.py'>

In [5]:
from wrappers import FrameSkip
print("FrameSkip OK")

FrameSkip OK


In [6]:
# Importar tus módulos
from dqn_agent import DQNAgent
from dqn_cnn import *
from wrappers import *
import train_dqn

print("✓ Módulos importados exitosamente")
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

✓ Módulos importados exitosamente
GPU disponible: True
GPU: Tesla T4


In [7]:
import importlib
import dqn_agent
importlib.reload(dqn_agent)

# 🔍 Verificar que estás usando el código correcto
from dqn_agent import ReplayBuffer
import inspect

print(inspect.getsource(ReplayBuffer))

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((
            state.astype(np.uint8),
            action,
            reward,
            next_state.astype(np.uint8),
            done
        ))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)

        return (
            np.array(states, dtype=np.uint8),
            np.array(actions, dtype=np.int64),
            np.array(rewards, dtype=np.float32),
            np.array(next_states, dtype=np.uint8),
            np.array(dones, dtype=np.float32),
        )

    def __len__(self):
        return len(self.buffer)



In [8]:
import inspect
print(inspect.getsource(train_dqn.train))

def train():
    env = gym.make("ALE/Pacman-v5", render_mode=None)
    env = FrameSkip(env, skip=4)
    env = PreprocessFrame(env)
    env = FrameStack(env, k=4)

    n_actions = env.action_space.n
    agent = DQNAgent(n_actions)

    episodes = 300

    for episode in range(episodes):
        state, _ = env.reset()
        total_reward = 0
        done = False

        # 🔥 contador REAL de pasos
        step_counter = 0

        while not done:
            action = agent.select_action(state)

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            # 🔥 REWARD SHAPING
            reward_shaped = reward / 10.0

            if done:
                reward_shaped -= 50

            reward_shaped += 0.1

            if reward > 0:
                reward_shaped += 2

            # 🔥 CLIPPING FINAL
            reward_shaped = np.clip(reward_shaped, -1, 1)

            # Guardar experiencia
            agent.memory.pus

In [9]:
# Ejecutar el entrenamiento con GPU
print("🚀 Iniciando entrenamiento con GPU...")
print("=" * 50)

# Ejecutar el script de entrenamiento
train_dqn.train()

print("=" * 50)
print("✅ Entrenamiento completado")
print("📁 Modelo guardado en: results/dqn_pacman_improved.pth")

🚀 Iniciando entrenamiento con GPU...
Episode 1 | Reward: 14.00 | Epsilon: 1.000 | Memory: 87
Episode 2 | Reward: 12.00 | Epsilon: 1.000 | Memory: 182
Episode 3 | Reward: 12.00 | Epsilon: 1.000 | Memory: 291
Episode 4 | Reward: 15.00 | Epsilon: 1.000 | Memory: 382
Episode 5 | Reward: 18.00 | Epsilon: 1.000 | Memory: 479
Episode 6 | Reward: 15.00 | Epsilon: 1.000 | Memory: 603
Episode 7 | Reward: 14.00 | Epsilon: 1.000 | Memory: 708
Episode 8 | Reward: 32.00 | Epsilon: 1.000 | Memory: 830
Episode 9 | Reward: 19.00 | Epsilon: 1.000 | Memory: 933
Episode 10 | Reward: 53.00 | Epsilon: 0.991 | Memory: 1071
Episode 11 | Reward: 17.00 | Epsilon: 0.980 | Memory: 1165
Episode 12 | Reward: 11.00 | Epsilon: 0.968 | Memory: 1261
Episode 13 | Reward: 7.00 | Epsilon: 0.959 | Memory: 1340
Episode 14 | Reward: 21.00 | Epsilon: 0.946 | Memory: 1455
Episode 15 | Reward: 23.00 | Epsilon: 0.932 | Memory: 1571
Episode 16 | Reward: 22.00 | Epsilon: 0.920 | Memory: 1671
Episode 17 | Reward: 35.00 | Epsilon: 0

: 

: 

: 

In [ ]:
# Verificar que el modelo se guardó correctamente
import os

model_path = "results/dqn_pacman_improved.pth"
if os.path.exists(model_path):
    model_size = os.path.getsize(model_path) / (1024 * 1024)  # MB
    print(f"✅ Modelo guardado exitosamente: {model_size:.2f} MB")
    print(f"📍 Ubicación: {model_path}")

    # Mostrar archivos en results/
    print("\n📁 Contenido de la carpeta results:")
    for file in os.listdir("results"):
        file_path = os.path.join("results", file)
        size = os.path.getsize(file_path) / (1024 * 1024)
        print(f"  - {file}: {size:.2f} MB")
else:
    print("❌ Error: El modelo no se guardó correctamente")